In [0]:
%run ./ngo_medallion_Mohamed_Al_Sharqawy/00_config

In [0]:
from pyspark.sql import functions as F

# تحميل الجداول من طبقة Silver
cases = spark.table(silver_cases)
services = spark.table(silver_services)

# --- 1. جدول الخدمات اليومية (Daily Services KPI) ---
daily_services = (
    services
    .groupBy("service_date")
    .agg(
        F.countDistinct("service_id").alias("total_services"),
        F.countDistinct("case_id").alias("unique_cases_served")
    )
    .orderBy("service_date")
)
daily_services.write.format("delta").mode("overwrite").saveAsTable(gold_daily_services)

# --- 2. جدول نظرة شاملة على الحالة (Case 360) ---
case_360 = (
    cases.alias("c")
    .join(services.alias("s"), F.col("c.case_id") == F.col("s.case_id"), "left")
    .groupBy("c.case_id")
    .agg(
        F.countDistinct("s.service_id").alias("total_services_received"),
        F.min("s.service_date").alias("first_service_date"),
        F.max("s.service_date").alias("last_service_date")
    )
)
case_360.write.format("delta").mode("overwrite").saveAsTable(gold_case_360)

# عرض النتيجة
display(daily_services)

service_date,total_services,unique_cases_served
2023-12-27,3,3
2023-12-28,8,8
2023-12-29,11,11
2023-12-30,14,14
2023-12-31,18,18
2024-01-01,8,8
2024-01-02,18,18
2024-01-03,26,26
2024-01-04,28,28
2024-01-05,28,27
